# 03s — Build the surrogate dataset

For each clear-sky day `d0` with a cached WGAST output, assemble the 10-day
window `d-10..d-1` of (S2 + LS + optical mask + past WGAST + WGAST mask) plus
DEM, per-day daily weather, and the target-day forecast block, then save one
`.npz` per sample + a parquet manifest.

**Splits are city-based** (ROADMAP §3.7) — defined once in `cities.py::EXPERIMENT`:

| split | cities | why |
|---|---|---|
| train | Istanbul + Orléans | full year 2022 |
| val   | Rome  | cross-city val (Orléans is WGAST's own training city) |
| test  | Cairo | held-out stress test, untouched until the end |

After all cities are built, `fit_stats()` computes per-channel mean/std **on
train rows only** (val/test stats would be a leak) and writes `stats.npz` next
to the combined manifest. Set `WGAST_DATA_ROOT` to relocate the data directory.

In [1]:
# --- Configuration -------------------------------------------------------------
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import Resampling, reproject

from cities import DATA_ROOT, EXPERIMENT, split_of

WINDOW = 10

# Build every experiment city that has data on disk; edit to a subset if needed.
CITIES_TO_BUILD = list(EXPERIMENT)

print("data root:", DATA_ROOT)
print("cities:", {c: EXPERIMENT[c]["split"] for c in CITIES_TO_BUILD})

data root: /Users/ihsanbolum/WGAST_w--Prediction/tutorials/data/secondary
cities: {'Istanbul': 'train', 'Orleans': 'train', 'Rome': 'val', 'Cairo': 'test'}


In [2]:
# --- Filename parsers & raster IO ------------------------------------------------
def _p_wgast(p): return datetime.strptime(p.stem.split("_")[1], "%Y%m%d").date()
def _p_s2(p):    return datetime.strptime(p.stem[:8], "%Y%m%d").date()
def _p_ls(p):    return datetime.strptime(p.stem.split("_")[-1], "%Y%m%d").date()

def _index(paths, parse):
    out = {}
    for p in paths:
        out.setdefault(parse(p), []).append(p)
    return out


def _read_aligned(src_path, ref, bands):
    """Read `bands` from src_path, reproject onto the WGAST reference grid."""
    with rasterio.open(src_path) as src:
        out = np.zeros((len(bands), ref["height"], ref["width"]), dtype=np.float32)
        for i, b in enumerate(bands):
            reproject(
                source=rasterio.band(src, b), destination=out[i],
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=ref["transform"], dst_crs=ref["crs"],
                resampling=Resampling.bilinear,
            )
    return out

def _stack_date(paths, ref, bands, n_bands):
    if not paths:
        return np.zeros((n_bands, ref["height"], ref["width"]), dtype=np.float32)
    arrs = [_read_aligned(p, ref, bands) for p in paths]
    # Mask-aware mosaic: later scenes fill only where they have valid (nonzero)
    # pixels. np.maximum would overwrite valid NEGATIVE index values (water,
    # built-up NDVI/NDWI/NDBI) with nodata zeros on multi-tile cities.
    out = arrs[0]
    for a in arrs[1:]:
        valid = np.any(a != 0, axis=0)
        out[:, valid] = a[:, valid]
    return out

def _opt_mask(stack):
    return (np.any(stack != 0, axis=0)).astype(np.float32)

def _ref_grid(wgast_path):
    with rasterio.open(wgast_path) as s:
        return dict(crs=s.crs, transform=s.transform, height=s.height, width=s.width)

In [3]:
# --- Per-d0 sample ----------------------------------------------------------------
def build_sample(d0, ref, wgast_idx, s2_idx, ls_idx, weather, target_weather, dem):
    H, W = ref["height"], ref["width"]
    s2s, lss, wgs, opt_m, wg_m = [], [], [], [], []

    for k in range(1, WINDOW + 1):
        dk = d0 - timedelta(days=k)
        s2 = _stack_date(s2_idx.get(dk, []), ref, [1, 2, 3], 3)
        ls = _stack_date(ls_idx.get(dk, []), ref, [2, 3, 4], 3)   # skip LS band 1 (LST)
        opt = ((_opt_mask(s2) + _opt_mask(ls)) > 0).astype(np.float32)

        if dk in wgast_idx:
            wg = _read_aligned(wgast_idx[dk][0], ref, [1])[0]
            wm = np.ones((H, W), dtype=np.float32)
        else:
            wg = np.zeros((H, W), dtype=np.float32)
            wm = np.zeros((H, W), dtype=np.float32)

        s2s.append(s2); lss.append(ls); wgs.append(wg); opt_m.append(opt); wg_m.append(wm)

    # N=1 filters
    if not any(m.any() for m in opt_m): return None
    if not any(m.any() for m in wg_m):  return None

    # spatial stack: 10 * (3 S2 + 3 LS + 1 opt + 1 WGAST + 1 WGmask) + 1 DEM = 91
    slots = []
    for s2, ls, om, wg, wm in zip(s2s, lss, opt_m, wgs, wg_m):
        slots += [s2, ls, om[None], wg[None], wm[None]]
    spatial = np.concatenate(slots + [dem], axis=0)

    # scalars: 17 weather * 10 days + 15 target-day forecast + sin/cos(doy) + elev
    wx = []
    for k in range(1, WINDOW + 1):
        dk = d0 - timedelta(days=k)
        row = weather.loc[weather.index.date == dk]
        wx.append(row.iloc[0].values if len(row) else np.zeros(weather.shape[1]))
    wx = np.concatenate(wx).astype(np.float32)
    # Target-day forecast block (ROADMAP §3.1): NWP weather for d0 itself —
    # at inference this slot is filled by the live d+1 forecast.
    trow = target_weather.loc[target_weather.index.date == d0]
    wx_t = (trow.iloc[0].values if len(trow)
            else np.zeros(target_weather.shape[1])).astype(np.float32)
    doy = d0.timetuple().tm_yday
    season = np.array([np.sin(2 * np.pi * doy / 365), np.cos(2 * np.pi * doy / 365)], dtype=np.float32)
    elev = np.array([float(dem.mean())], dtype=np.float32)  # region-level scalar (ROADMAP §3.1)
    scalars = np.concatenate([wx, wx_t, season, elev])

    target = _read_aligned(wgast_idx[d0][0], ref, [1])[0]
    return spatial, scalars, target

In [4]:
# --- Build all samples for one city -------------------------------------------------
def build_city(city):
    cap = city.capitalize()
    wgast_dir = DATA_ROOT / "wgast_cache" / city.lower()
    # Per-day window acquisitions (01s window-export cell) — NOT the t0 reference
    # scenes, which only exist on a few dates per city (ROADMAP §10 C1).
    s2_dir    = DATA_ROOT / "raw" / f"Sentinel2_window_{cap}"
    ls_dir    = DATA_ROOT / "raw" / f"Landsat8_window_{cap}"
    dem_path  = DATA_ROOT / "raw" / f"DEM_{cap}.tif"
    weather_p = DATA_ROOT / f"weather_{cap}_daily.parquet"
    target_wp = DATA_ROOT / f"weather_{cap}_target_daily.parquet"
    out_dir   = DATA_ROOT / f"samples_{city.lower()}"

    missing = [p for p in (wgast_dir, s2_dir, ls_dir, dem_path, weather_p, target_wp)
               if not p.exists()]
    if missing:
        print(f"[WARN] {city}: NOT built, missing inputs:")
        for p in missing:
            print("   -", p)
        return None

    out_dir.mkdir(exist_ok=True)
    for stale in out_dir.glob("sample_*.npz"):  # keep manifest in sync with disk
        stale.unlink()

    wgast_idx = _index(sorted(wgast_dir.glob("wgast_*.tif")), _p_wgast)
    s2_idx    = _index(sorted(s2_dir.glob("*.tif")),          _p_s2)
    ls_idx    = _index(sorted(ls_dir.glob("LC08_*.tif")),     _p_ls)
    weather   = pd.read_parquet(weather_p)
    weather.index = pd.to_datetime(weather.index)
    if weather.index.tz is not None:
        raise ValueError(
            f"{weather_p} has a timezone-aware index (fetched before the C2 "
            "timezone fix) -- its daily rows are shifted by one local day. "
            "Refetch weather (01s, weather cell) first."
        )
    target_weather = pd.read_parquet(target_wp)
    target_weather.index = pd.to_datetime(target_weather.index)

    split = split_of(city)
    rows = []
    for d0, paths in sorted(wgast_idx.items()):
        ref = _ref_grid(paths[0])
        dem = _read_aligned(dem_path, ref, [1])
        sample = build_sample(d0, ref, wgast_idx, s2_idx, ls_idx, weather, target_weather, dem)
        if sample is None:
            print(f"  skip {d0} (filters)"); continue
        spatial, scalars, target = sample
        out = out_dir / f"sample_{d0:%Y%m%d}.npz"
        np.savez_compressed(out, spatial=spatial, scalars=scalars, target=target)
        rows.append(dict(city=city.lower(), d0=str(d0), split=split, path=str(out)))
        print(f"  kept {d0}  spatial={spatial.shape}  scalars={scalars.shape}")

    manifest = out_dir / "manifest.parquet"
    pd.DataFrame(rows).to_parquet(manifest)
    print(f"[{city}] {len(rows)}/{len(wgast_idx)} samples written -> {out_dir}")
    return manifest

In [5]:
# --- Per-channel z-score stats -------------------------------------------------------
def fit_stats(manifest_path):
    """Single pass over the TRAIN rows of the manifest (val/test stats would be
    a leak). Per-channel mean/std for spatial (C,H,W) computed over VALID
    (nonzero) pixels only — zero-filled missing slots otherwise drag a channel's
    stats toward its missingness rate instead of its physical scale (§10 H2).
    Per-element mean/std for scalars (S,), scalar mean/std for target."""
    df = pd.read_parquet(manifest_path)
    if "path" not in df.columns or len(df) == 0:
        raise ValueError(
            f"No samples in {manifest_path} -- every d0 was filtered out. "
            "Check that the optical (S2/LS) directories exist and overlap the "
            "WGAST window dates before fitting stats."
        )
    if "split" in df.columns and (df["split"] == "train").any():
        df = df[df["split"] == "train"]
    else:
        print("[WARN] no train rows found -- fitting stats on ALL rows")

    sp_sum = sp_sq = sp_n = None
    sc_sum = sc_sq = sc_n = None
    tg_sum = tg_sq = tg_n = 0.0

    for path in df["path"]:
        d = np.load(path)
        sp, sc, tg = d["spatial"], d["scalars"], d["target"]

        # spatial: per-channel sums over valid (nonzero) pixels
        if sp_sum is None:
            sp_sum = np.zeros(sp.shape[0], dtype=np.float64)
            sp_sq  = np.zeros(sp.shape[0], dtype=np.float64)
            sp_n   = np.zeros(sp.shape[0], dtype=np.float64)
        valid = sp != 0
        sp64 = sp.astype(np.float64)
        sp_sum += np.where(valid, sp64, 0).sum(axis=(1, 2))
        sp_sq  += np.where(valid, sp64 ** 2, 0).sum(axis=(1, 2))
        sp_n   += valid.sum(axis=(1, 2))

        # scalars: per-element across samples
        if sc_sum is None:
            sc_sum = np.zeros(sc.shape[0], dtype=np.float64)
            sc_sq  = np.zeros(sc.shape[0], dtype=np.float64)
            sc_n   = 0
        sc_sum += sc
        sc_sq  += sc.astype(np.float64) ** 2
        sc_n   += 1

        tg_sum += float(tg.sum())
        tg_sq  += float((tg.astype(np.float64) ** 2).sum())
        tg_n   += tg.size

    sp_n    = np.maximum(sp_n, 1)  # all-missing channels -> mean 0, std 1
    sp_mean = (sp_sum / sp_n).astype(np.float32)
    sp_std  = np.sqrt(np.maximum(sp_sq / sp_n - sp_mean ** 2, 1e-12)).astype(np.float32)
    sp_std  = np.where(sp_std <= 1e-6, 1.0, sp_std).astype(np.float32)  # constant channels -> no scaling

    sc_mean = (sc_sum / sc_n).astype(np.float32)
    sc_std  = np.sqrt(np.maximum(sc_sq / sc_n - sc_mean ** 2, 1e-12)).astype(np.float32)
    sc_std  = np.where(sc_std <= 1e-6, 1.0, sc_std).astype(np.float32)

    tg_mean = np.float32(tg_sum / tg_n)
    tg_std  = np.float32(max(np.sqrt(tg_sq / tg_n - float(tg_mean) ** 2), 1e-6))

    stats_path = Path(manifest_path).parent / "stats.npz"
    np.savez(stats_path,
             spatial_mean=sp_mean, spatial_std=sp_std,
             scalars_mean=sc_mean, scalars_std=sc_std,
             target_mean=tg_mean, target_std=tg_std)
    print(f"stats written -> {stats_path}")
    print(f"  spatial std range: {sp_std.min():.3g} .. {sp_std.max():.3g}")
    print(f"  scalars std range: {sc_std.min():.3g} .. {sc_std.max():.3g}")
    print(f"  target mean={tg_mean:.4g}  std={tg_std:.4g}")
    return stats_path

In [6]:
# --- Run: build every city, combine, fit stats ---------------------------------------
built = []
for c in CITIES_TO_BUILD:
    print(f"\n=== {c} ({EXPERIMENT[c]['split']}) ===")
    m = build_city(c)
    if m is not None:
        built.append(m)

if not built:
    raise RuntimeError(f"No city could be built -- check {DATA_ROOT}")

# samples_all/manifest.parquet is THE canonical manifest: 04s/05s read only this.
out_dir = DATA_ROOT / "samples_all"
out_dir.mkdir(exist_ok=True)
combined = pd.concat([pd.read_parquet(m) for m in built], ignore_index=True)
combined_path = out_dir / "manifest.parquet"
combined.to_parquet(combined_path)
print(f"\ncombined manifest: {len(combined)} rows -> {combined_path}")

fit_stats(combined_path)

print("\nsplit counts:")
print(combined.groupby(["split", "city"]).size())


=== Istanbul (train) ===
[WARN] Istanbul: NOT built, missing inputs:
   - /Users/ihsanbolum/WGAST_w--Prediction/tutorials/data/secondary/wgast_cache/istanbul
   - /Users/ihsanbolum/WGAST_w--Prediction/tutorials/data/secondary/raw/Sentinel2_window_Istanbul
   - /Users/ihsanbolum/WGAST_w--Prediction/tutorials/data/secondary/raw/Landsat8_window_Istanbul
   - /Users/ihsanbolum/WGAST_w--Prediction/tutorials/data/secondary/weather_Istanbul_daily.parquet
   - /Users/ihsanbolum/WGAST_w--Prediction/tutorials/data/secondary/weather_Istanbul_target_daily.parquet

=== Orleans (train) ===
  skip 2022-03-07 (filters)
  kept 2022-03-08  spatial=(91, 1200, 1200)  scalars=(188,)
  kept 2022-03-09  spatial=(91, 1200, 1200)  scalars=(188,)
  kept 2022-03-10  spatial=(91, 1200, 1200)  scalars=(188,)
  kept 2022-03-14  spatial=(91, 1200, 1200)  scalars=(188,)
  kept 2022-03-19  spatial=(91, 1200, 1200)  scalars=(188,)
  kept 2022-03-21  spatial=(91, 1200, 1200)  scalars=(188,)
  kept 2022-03-22  spatial=(